# Break-Even Cost Accounting

Computes all cost inputs needed for the thesis break-even analysis comparing
LLaMA 3.1-8B and Ministral-8B (fine-tuned students) against a GPT-5 baseline.

**Sections**
1. Find silver dataset CSV
2. Token counts (tiktoken `cl100k_base`)
3. GPT baseline cost per ticket (self-consistency ×5)
4. Fixed costs per model (labeling + training)
5. Variable inference costs (from logged `latency_ms`)
6. Summary dict

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import tiktoken

# ── Locate project root (same logic as results_main.ipynb) ────────────────────
def _find_project_root() -> Path:
    candidates = []
    if '__vsc_ipynb_file__' in globals():
        p = Path(globals()['__vsc_ipynb_file__']).resolve().parents[2]
        if (p / 'data').is_dir() and (p / 'Notebooks').is_dir():
            candidates.append(p)
    for p in [Path.cwd()] + list(Path.cwd().parents):
        if (p / 'data' / 'results').is_dir() and (p / 'Notebooks').is_dir():
            candidates.append(p)
            break
    for p in [Path.cwd()] + list(Path.cwd().parents):
        candidate = p / 'Users' / 'josta' / 'Thesis_IT_TicketClassification'
        if candidate.is_dir() and (candidate / 'data').is_dir():
            candidates.append(candidate)
            break
    if candidates:
        return candidates[0]
    raise RuntimeError(
        "Cannot locate project root.\n"
        "Set ROOT manually: ROOT = Path('/absolute/path/to/Thesis_IT_TicketClassification')"
    )

ROOT = Path(_find_project_root())
DATA = ROOT / 'data'
RES  = DATA / 'results'
print(f'ROOT : {ROOT}')
print(f'RES  : {RES}  (exists={RES.is_dir()})')

---
## Task 1 — Find the silver dataset CSV

In [ ]:
# Search data/ for CSVs with both 'text' and 'label' columns
candidates = []
for csv_path in sorted(DATA.rglob('*.csv')):
    try:
        header = pd.read_csv(csv_path, nrows=0)
        cols = header.columns.tolist()
        if 'text' in cols and 'label' in cols:
            n = sum(1 for _ in open(csv_path)) - 1  # fast row count
            candidates.append({'path': str(csv_path.relative_to(ROOT)), 'rows': n, 'columns': cols})
    except Exception:
        pass

print(f'Found {len(candidates)} candidate(s):\n')
for c in sorted(candidates, key=lambda x: -x['rows']):
    print(f"  {c['rows']:>6,} rows  {c['path']}")
    print(f"            cols: {c['columns']}")
    print()

# ── Select the primary labeling dataset ───────────────────────────────────────
# prediction_results_cisc.csv (26 331 rows) is the FULL set of tickets sent to
# GPT-4o for self-consistency labeling.  silver_standard_dataset.csv is the
# quality-filtered subset used for training/evaluation.
#
# For cost accounting we want the full set (everything GPT was billed for).
SILVER_PATH = DATA / 'labeled' / 'CISC_Fixed' / 'prediction_results_cisc.csv'
print(f'Selected: {SILVER_PATH.relative_to(ROOT)}')

---
## Task 2 — Compute average token counts

> **Note on column mapping**
> The task spec described the `label` column as containing the full CoT output
> (`Reasoning: ...\nTag: ...`).  In this dataset the columns are split:
> - `label`     — short classification tag only  
> - `reasoning` — full GPT output (tag + reasoning + confidence scores)  
>
> We tokenize `reasoning` as `output_tokens` because that is what GPT actually
> generated and was billed for.

In [ ]:
df = pd.read_csv(SILVER_PATH)
print(f'Loaded {len(df):,} rows, columns: {df.columns.tolist()}')

# Drop rows where text or reasoning is missing
before = len(df)
df = df.dropna(subset=['text', 'reasoning'])
print(f'After dropping NaN text/reasoning: {len(df):,} rows ({before - len(df)} dropped)')

In [ ]:
enc = tiktoken.get_encoding('cl100k_base')

print('Tokenizing input (text column) …')
df['input_tokens']  = df['text'].astype(str).apply(lambda t: len(enc.encode(t)))

print('Tokenizing output (reasoning column = full CoT) …')
df['output_tokens'] = df['reasoning'].astype(str).apply(lambda t: len(enc.encode(t)))

print('Done.')

In [ ]:
mean_input   = df['input_tokens'].mean()
mean_output  = df['output_tokens'].mean()
med_input    = df['input_tokens'].median()
med_output   = df['output_tokens'].median()
p95_input    = df['input_tokens'].quantile(0.95)
p95_output   = df['output_tokens'].quantile(0.95)
total_input  = df['input_tokens'].sum()
total_output = df['output_tokens'].sum()

print('=== Token Statistics (cl100k_base) ===')
print(f'  Mean   input  tokens/ticket : {mean_input:>10.1f}')
print(f'  Mean   output tokens/ticket : {mean_output:>10.1f}')
print(f'  Median input  tokens/ticket : {med_input:>10.1f}')
print(f'  Median output tokens/ticket : {med_output:>10.1f}')
print(f'  P95    input  tokens/ticket : {p95_input:>10.1f}')
print(f'  P95    output tokens/ticket : {p95_output:>10.1f}')
print(f'  Total  input  tokens        : {total_input:>12,}')
print(f'  Total  output tokens        : {total_output:>12,}')

---
## Task 3 — GPT baseline cost per ticket

Rates (GPT-4o / GPT-5 tier assumed):
- Input : **\$1.25 / 1M tokens**
- Output: **\$10.00 / 1M tokens**
- Self-consistency multiplier: **×5 paths per ticket**

In [ ]:
INPUT_RATE  = 1.25   # USD per 1M tokens
OUTPUT_RATE = 10.00  # USD per 1M tokens
SC_MULT     = 5      # self-consistency paths

cost_per_ticket_gpt = (
    mean_input  * INPUT_RATE  / 1_000_000 +
    mean_output * OUTPUT_RATE / 1_000_000
) * SC_MULT

total_labeling_cost_usd = (
    total_input  * INPUT_RATE  / 1_000_000 +
    total_output * OUTPUT_RATE / 1_000_000
) * SC_MULT

print('=== GPT Baseline Cost ===')
print(f'  Cost per ticket (GPT ×5)    : ${cost_per_ticket_gpt:.6f}')
print(f'  Total labeling cost (all)   : ${total_labeling_cost_usd:>10.2f}')

---
## Task 4 — Fixed costs per model

The **labeling cost is shared** — paid once regardless of how many student models are trained.

| Component | Value |
|---|---|
| Shared labeling (GPT ×5) | computed above |
| Llama 3.1-8B training    | 3.9 h × \$4.00/h |
| Ministral-8B training    | 18.5 h × \$4.00/h |

In [ ]:
GPU_RATE_PER_HR = 4.00  # USD / hr  (Standard_NC24ads_A100_v4)

LLAMA_TRAIN_HRS   = 3.9
MISTRAL_TRAIN_HRS = 18.5

shared_labeling_cost_usd  = total_labeling_cost_usd
llama_training_cost_usd   = LLAMA_TRAIN_HRS   * GPU_RATE_PER_HR
mistral_training_cost_usd = MISTRAL_TRAIN_HRS * GPU_RATE_PER_HR

fixed_cost_llama   = shared_labeling_cost_usd + llama_training_cost_usd
fixed_cost_mistral = shared_labeling_cost_usd + mistral_training_cost_usd

print('=== Fixed Costs ===')
print(f'  Shared labeling cost        : ${shared_labeling_cost_usd:>10.2f}')
print()
print(f'  Llama   training cost       : ${llama_training_cost_usd:>10.2f}  ({LLAMA_TRAIN_HRS} h × ${GPU_RATE_PER_HR:.2f}/h)')
print(f'  Mistral training cost       : ${mistral_training_cost_usd:>10.2f}  ({MISTRAL_TRAIN_HRS} h × ${GPU_RATE_PER_HR:.2f}/h)')
print()
print(f'  Fixed cost — Llama          : ${fixed_cost_llama:>10.2f}  (labeling + training)')
print(f'  Fixed cost — Mistral        : ${fixed_cost_mistral:>10.2f}  (labeling + training)')

---
## Task 5 — Variable inference costs

Latency was logged (in ms) in the eval result JSON files for the fine-tuned models.
Inference is assumed to run on a **Standard_NC24ads_A100_v4** at **\$4.00/hr**.

$$\text{variable\_cost\_per\_ticket} = \frac{\text{mean\_latency\_ms}}{1000 \times 3600} \times 4.00$$

In [ ]:
INFERENCE_GPU_RATE = 4.00  # USD / hr

latency_files = {
    'llama':   RES / 'qlora_llama.json',
    'mistral': RES / 'qlora_mistral.json',
}

variable_cost = {}
mean_latency  = {}

for model, path in latency_files.items():
    if not path.exists():
        print(f'[{model}] latency data not found — will assume $0 on-prem variable cost')
        variable_cost[model] = 0.0
        mean_latency[model]  = None
        continue

    with open(path) as f:
        d = json.load(f)
    df_lat = pd.DataFrame(d['data'], columns=d['columns'])

    if 'latency_ms' not in df_lat.columns:
        print(f'[{model}] latency_ms column missing — assuming $0')
        variable_cost[model] = 0.0
        mean_latency[model]  = None
        continue

    ml = df_lat['latency_ms'].mean()
    vc = (ml / 1000 / 3600) * INFERENCE_GPU_RATE
    mean_latency[model]  = ml
    variable_cost[model] = vc

    print(f'[{model}]')
    print(f'  Rows                        : {len(df_lat):,}')
    print(f'  Mean latency                : {ml:.1f} ms')
    print(f'  Variable cost / ticket      : ${vc:.8f}')

variable_cost_per_ticket_llama_usd   = variable_cost['llama']
variable_cost_per_ticket_mistral_usd = variable_cost['mistral']

---
## Task 6 — Summary

In [ ]:
summary = {
    'n_tickets_labeled':                   len(df),
    'mean_input_tokens':                   round(mean_input,  2),
    'mean_output_tokens':                  round(mean_output, 2),
    'cost_per_ticket_gpt_usd':             round(cost_per_ticket_gpt,                 8),
    'shared_labeling_cost_usd':            round(shared_labeling_cost_usd,             2),
    'llama_training_cost_usd':             round(llama_training_cost_usd,              2),
    'mistral_training_cost_usd':           round(mistral_training_cost_usd,            2),
    'fixed_cost_llama_usd':                round(fixed_cost_llama,                     2),
    'fixed_cost_mistral_usd':              round(fixed_cost_mistral,                   2),
    'variable_cost_per_ticket_llama_usd':  round(variable_cost_per_ticket_llama_usd,  8),
    'variable_cost_per_ticket_mistral_usd':round(variable_cost_per_ticket_mistral_usd,8),
}

print('=== COST ACCOUNTING SUMMARY ===')
print('{')
for k, v in summary.items():
    print(f'  {repr(k)}: {v},')
print('}')

---
## Task 7 — PoC (current production) variable cost per ticket

The PoC (`autotagging_servicenow.ipynb`) uses **GPT-5** (Azure OpenAI) with:
- A single inference per ticket (no self-consistency)
- A fixed prompt overhead: system instructions + full category/scenario lists + 7 few-shot examples
- Output: tag only (no chain-of-thought reasoning)

Rates: same GPT-5 tier ($1.25/M input · $10.00/M output · ×**1** call per ticket)

In [ ]:
import tiktoken
enc = tiktoken.get_encoding('cl100k_base')

# ── Reconstruct the PoC prompt template ──────────────────────────────────────
POC_HIERARCHICAL_CATEGORIES = [
    ('1a MV (Mobile voice),', '2a Main product'), ('1a MV (Mobile voice),', '2a Smartwatch'),
    ('1a MV (Mobile voice),', '2a Datasharingcard'), ('1a MV (Mobile voice),', '2a Nummerportering'),
    ('1b MBB', '2b Main product'), ('1b MBB', '2b Datasharingcard'),
    ('1b MBB', '2b Router'), ('1b MBB', '2b Smart-sim'),
    ('1c TV', '2c COAX'), ('1c TV', '2c Fiber'), ('1c TV', '2c DSL'), ('1c TV', '2c OTT'),
    ('1d Broadband', '2d COAX'), ('1d Broadband', '2d Fiber'), ('1d Broadband', '2d DSL'),
    ('1e VoIP', '2e COAX'), ('1e VoIP', '2e Fiber'), ('1e VoIP', '2e DSL'),
    ('1e VoIP', '2e Fastnet på mobil'), ('1f PSTN', '2f Main product'), ('1f PSTN', '2f Add-on'),
    ('1g Self service', '2g Mit YouSee'), ('1g Self service', '2g YouSee Music'),
    ('1g Self service', '2g Mit Internet'), ('1g Self service', '2g YS-play'),
    ('1g Self service', '2g Webmail'), ('1 Misc incidents', '2 Other'),
]
POC_SCENARIOS = [
    "3 App","3 Barring","3 Billing/Invoices","3 Change ownership","3 Click & Collect",
    "3 CPR issues in Dawn","3 Credit check","3 CSRD","3 Fraud","3 Login issues",
    "3 Missing rights to access","3 Mix-tv","3 product status","3 relocate","3 onsite technician",
    "3 line technician","3 Port in","3 Port out","3 payment method","3 technical issues",
    "3 Order activation","3 Order confirmation","3 Performance issues","3 Power of attorney",
    "3 Price and campaigns","3 Proof of purchase","3 Quote error","3 Return/replace","3 Shipment",
    "3 Sikkerhedspakke","3 Stock","3 Termination","3 Third party","3 Tickets","3 User error","3 Web",
]
POC_FEW_SHOTS = \"\"\"Description: The customer was supposed to have their internet and TV activated today, but the order has not gone through yet. It still shows as \\"Activation in progress.\\"
Tag: (1d Broadband, 2d Fiber), 3 product status


Description: Wifi Health is Excellent despite high dBm, is this correct? Expected Result:The estimated quality of the wifi should represent the high dBm  
Tag: (1d Broadband, 2d Fiber), 3 Performance issues


Description: I get an error message that I have to use the app in Denmark before I can use it abroad. ExpectedBehaviour:Login to use yousee play without any problem.StepsToReproduce:Playback error on Yousee play  
Tag: (1g Self Service, 2g Yousee TV Play), 3 Login issues


Description: The router does not have VoIP configuration. The router has been reset.  Expected Result:  
The router should receive VoIP configuration when customers have VoIP subscriptions.  
Tag: (1e VoIP, 2e COAX), 3 technical issues


Description: The customer is experiencing problems logging into Disney+. After entering the email address and pressing \\"continue\\" the error \\"an error occurred\\" appears on the screen.  
I have informed the customer that the app is integrated and that I will let our backend know.  
Tag: (1c TV, 2c Fiber), 3 Login issues


Description: Cannot proceed with the order, Fiber should have been activated on the 16th, but the basket and order are at a standstill.
Tag: (1d Broadband, 2d Fiber, 3 Order activation\"


Description: Hiper has requested the SBBU rebooking, but Dawn is not allowing it  
Tag: (1d Broadband, 2d COAX), 3 Missing rights to access\"\"\"

# Static prompt portion (everything except the variable ticket description)
poc_static_prompt = f\"\"\"
    You are an expert in Nuuday telecommunication service classfication, top notch at categorizing Service Now for different brands such as YouSee, Telmore, Eesy, and Hiper customer tickets with super high accuracy.
    Your task is to categorize service descriptions into the correct hierarchical category.
    
    Available categories: {POC_HIERARCHICAL_CATEGORIES}
    available scenarios: {POC_SCENARIOS}
    
    Rules:
    1. Return EXACTLY the format: (Category, Subcategory), Scenarios
    2. Choose the most appropriate category and subcategory and scenarios based on the description
    3. If you\'re unsure or if the description doesn\'t fit any category, return: (1 Misc incidents, 2 Other), Scenarios
    4. Be consistent and accurate in your classifications

    Examples:
    {POC_FEW_SHOTS}

    Input Description: 
    Output Tag:\"\"\"

poc_static_tokens = len(enc.encode(poc_static_prompt))

# Mean output = tag only (measured from PoC_results.csv, 50 samples)
poc_results = pd.read_csv(DATA / 'labeled' / 'PoC' / 'PoC_results.csv')
poc_results['label_tokens'] = poc_results['label'].astype(str).apply(lambda t: len(enc.encode(t)))
mean_poc_output_tokens = poc_results['label_tokens'].mean()

# Mean input = static overhead + mean ticket text (from Task 2)
mean_poc_input_tokens = poc_static_tokens + mean_input  # mean_input from Task 2

poc_variable_cost_per_ticket = (
    mean_poc_input_tokens  * INPUT_RATE  / 1_000_000 +
    mean_poc_output_tokens * OUTPUT_RATE / 1_000_000
)  # no SC multiplier

print('=== PoC Cost Profile ===')
print(f'  Static prompt tokens (categories+scenarios+few-shots): {poc_static_tokens}')
print(f'  Mean ticket text tokens:                               {mean_input:.1f}')
print(f'  Mean total input  tokens/call:                         {mean_poc_input_tokens:.1f}')
print(f'  Mean output tokens/call (tag only):                    {mean_poc_output_tokens:.1f}')
print()
print(f'  Input  cost/ticket  : ${mean_poc_input_tokens  * INPUT_RATE  / 1_000_000:.6f}')
print(f'  Output cost/ticket  : ${mean_poc_output_tokens * OUTPUT_RATE / 1_000_000:.6f}')
print(f'  PoC variable cost/ticket (×1, no SC): ${poc_variable_cost_per_ticket:.6f}')


---
## Task 8 — Break-even analysis & plot

**Cost model:**
| | Fixed cost | Variable cost / ticket |
|---|---|---|
| PoC (GPT-5, no SC) | $0 | $0.001801 |
| Llama fine-tuned | $163.94 | $0.000813 |
| Mistral fine-tuned | $222.34 | $0.002095 |

> **Key insight:** Mistral inference is *more expensive* per ticket than the PoC, so it never breaks even on cost alone.
> Only Llama is cheaper per-inference, giving it a positive payoff slope.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# ── Cost parameters ──────────────────────────────────────────────────────────
poc_var   = poc_variable_cost_per_ticket          # from Task 7
llama_fix = fixed_cost_llama                       # from Task 4
llama_var = variable_cost_per_ticket_llama_usd    # from Task 5
mis_fix   = fixed_cost_mistral
mis_var   = variable_cost_per_ticket_mistral_usd

# ── Break-even points ─────────────────────────────────────────────────────────
be_llama_tickets   = llama_fix / (poc_var - llama_var)   # positive: Llama wins
be_mistral_tickets = mis_fix   / (poc_var - mis_var)     # negative: Mistral never wins

print(f'Savings per ticket vs PoC:')
print(f'  Llama   : ${poc_var - llama_var:.6f}/ticket  → break-even at {be_llama_tickets:,.0f} tickets')
print(f'  Mistral : ${poc_var - mis_var:.6f}/ticket  → {"NEVER (Mistral > PoC per ticket)" if be_mistral_tickets < 0 else f"break-even at {be_mistral_tickets:,.0f} tickets"}')
print()
for v in [5_000, 10_000, 20_000]:
    m = be_llama_tickets / v
    print(f'  Llama @ {v:>6,} tickets/month → break-even in {m:.1f} months ({m/12:.1f} years)')


In [ ]:
# ── Plot ─────────────────────────────────────────────────────────────────────
N_MAX  = 300_000
n      = np.linspace(0, N_MAX, 2000)

cost_poc     = poc_var   * n
cost_llama   = llama_fix + llama_var * n
cost_mistral = mis_fix   + mis_var   * n

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(n / 1_000, cost_poc,     color='#e74c3c', lw=2.5, label='PoC — GPT-5 (no fine-tuning, ×1)')
ax.plot(n / 1_000, cost_llama,   color='#2ecc71', lw=2.5, label='LLaMA fine-tuned (QLoRA)')
ax.plot(n / 1_000, cost_mistral, color='#3498db', lw=2.5, label='Mistral fine-tuned (QLoRA)', ls='--')

# ── Break-even vertical for Llama ─────────────────────────────────────────────
be_k = be_llama_tickets / 1_000
be_cost = poc_var * be_llama_tickets
ax.axvline(be_k, color='#2ecc71', lw=1.2, ls=':', alpha=0.8)
ax.annotate(
    f'Llama break-even\n{be_llama_tickets:,.0f} tickets\n(${be_cost:.0f})',
    xy=(be_k, be_cost), xytext=(be_k + 12, be_cost - 30),
    fontsize=8.5, color='#27ae60',
    arrowprops=dict(arrowstyle='->', color='#27ae60', lw=1),
)

# ── Monthly volume markers ────────────────────────────────────────────────────
monthly_volumes = [5_000, 10_000, 20_000]
colors_mv = ['#f39c12', '#8e44ad', '#1abc9c']
for v, col in zip(monthly_volumes, colors_mv):
    m_to_be = be_llama_tickets / v
    x_mark  = v / 1_000
    ax.axvline(x_mark, color=col, lw=1.0, ls='-.', alpha=0.6)
    ax.text(x_mark + 0.5, cost_mistral.max() * 0.97,
            f'{v//1000}k/mo\n→ Llama BE\n{m_to_be:.1f} mo',
            fontsize=7.5, color=col, va='top')

# ── Mistral annotation (never breaks even) ───────────────────────────────────
ax.text(0.97, 0.48,
        'Mistral per-ticket cost ($0.00210)\n> PoC per-ticket cost ($0.00180)\n→ no cost break-even',
        transform=ax.transAxes, fontsize=8, color='#2980b9',
        ha='right', va='top',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#2980b9', alpha=0.8))

ax.set_xlabel('Total tickets processed (thousands)', fontsize=11)
ax.set_ylabel('Cumulative cost (USD)', fontsize=11)
ax.set_title(
    'Break-even analysis: Student models vs. PoC (GPT-5, no fine-tuning)\n'
    'Fixed = labeling + training  |  Variable = per-ticket GPU inference cost',
    fontsize=12, pad=12
)
ax.legend(fontsize=10, loc='upper left')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'${y:.0f}'))
ax.set_xlim(0, N_MAX / 1_000)
ax.set_ylim(0)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT / 'fig_breakeven.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → fig_breakeven.png')


In [ ]:
# ── Monthly break-even summary table ─────────────────────────────────────────
rows = []
for v in [5_000, 10_000, 20_000]:
    poc_monthly     = poc_var * v
    llama_monthly   = llama_fix / v + llama_var     # amortized fixed + variable
    mis_monthly_var = mis_var * v                   # never recovers fixed
    be_months       = be_llama_tickets / v
    rows.append({
        'Monthly volume': f'{v:,}',
        'PoC monthly cost ($)':     f'${poc_monthly:,.2f}',
        'Llama monthly cost ($)':   f'${llama_var * v:,.2f}  (+${llama_fix:.0f} one-time)',
        'Mistral monthly cost ($)': f'${mis_var * v:,.2f}  (+${mis_fix:.0f} one-time)',
        'Llama BE (months)':        f'{be_months:.1f}',
        'Mistral BE':               'Never (higher var cost)',
    })

summary_df = pd.DataFrame(rows)
print('=== Monthly break-even summary ===')
print(summary_df.to_string(index=False))
